In [1]:
pip install pyspark

In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('DFApp').getOrCreate()

In [3]:
spark

In [4]:
iphones_RDD = spark.sparkContext.parallelize([
("XS", 2018, 5.65, 2.79, 6.24),
("XR", 2018, 5.94, 2.98, 6.84),
("X10", 2017, 5.65, 2.79, 6.13),
("8Plus", 2017, 6.23, 3.07, 7.12)
])
names = ['Model', 'Year', 'Height', 'Width', 'Weight']

In [5]:
iphone_df = spark.createDataFrame(iphones_RDD,names)

In [6]:
iphone_df.rdd.getNumPartitions()

2

In [7]:
iphone_df.show()

+-----+----+------+-----+------+
|Model|Year|Height|Width|Weight|
+-----+----+------+-----+------+
|   XS|2018|  5.65| 2.79|  6.24|
|   XR|2018|  5.94| 2.98|  6.84|
|  X10|2017|  5.65| 2.79|  6.13|
|8Plus|2017|  6.23| 3.07|  7.12|
+-----+----+------+-----+------+



In [8]:
iphone_df.schema

StructType([StructField('Model', StringType(), True), StructField('Year', LongType(), True), StructField('Height', DoubleType(), True), StructField('Width', DoubleType(), True), StructField('Weight', DoubleType(), True)])

In [9]:
iphone_df.printSchema()

root
 |-- Model: string (nullable = true)
 |-- Year: long (nullable = true)
 |-- Height: double (nullable = true)
 |-- Width: double (nullable = true)
 |-- Weight: double (nullable = true)



In [10]:
display(iphone_df)

DataFrame[Model: string, Year: bigint, Height: double, Width: double, Weight: double]

In [11]:
iphone_df.select('Model','Year').show()

+-----+----+
|Model|Year|
+-----+----+
|   XS|2018|
|   XR|2018|
|  X10|2017|
|8Plus|2017|
+-----+----+



In [12]:
iphone_df.filter(iphone_df.Year == 2018).show()

+-----+----+------+-----+------+
|Model|Year|Height|Width|Weight|
+-----+----+------+-----+------+
|   XS|2018|  5.65| 2.79|  6.24|
|   XR|2018|  5.94| 2.98|  6.84|
+-----+----+------+-----+------+



In [13]:
iphone_df.where('Width > 2.80').show()

+-----+----+------+-----+------+
|Model|Year|Height|Width|Weight|
+-----+----+------+-----+------+
|   XR|2018|  5.94| 2.98|  6.84|
|8Plus|2017|  6.23| 3.07|  7.12|
+-----+----+------+-----+------+



In [14]:
import pyspark.sql.functions as F
iphone_df.filter(F.col('Width') > 2.88).show()

+-----+----+------+-----+------+
|Model|Year|Height|Width|Weight|
+-----+----+------+-----+------+
|   XR|2018|  5.94| 2.98|  6.84|
|8Plus|2017|  6.23| 3.07|  7.12|
+-----+----+------+-----+------+



In [15]:
iphone_renamed = iphone_df.withColumnsRenamed({'Height':'ht','Weight':'wt'})

In [16]:
iphone_df.show()

+-----+----+------+-----+------+
|Model|Year|Height|Width|Weight|
+-----+----+------+-----+------+
|   XS|2018|  5.65| 2.79|  6.24|
|   XR|2018|  5.94| 2.98|  6.84|
|  X10|2017|  5.65| 2.79|  6.13|
|8Plus|2017|  6.23| 3.07|  7.12|
+-----+----+------+-----+------+



In [17]:
#iphone_filtered = iphone_df.select(F.expr('Model'),F.expr("cast(Year as double)as Year"),F.expr("Height as ht"),F.expr("Weight as wt"))
'''iphone_filtered = iphone_df.select(F.col('Model'),\
                                   F.col('Year').cast('double').alias('Year'),\
                                   F.col('Height').alias('Ht'),\
                                   F.col('Weight').alias('Wt'))'''
iphone_filtered = iphone_df.selectExpr("Model","cast(Year as double) as Year", "Height as ht","Weight as wt")

In [18]:
iphone_filtered.show()

+-----+------+----+----+
|Model|  Year|  ht|  wt|
+-----+------+----+----+
|   XS|2018.0|5.65|6.24|
|   XR|2018.0|5.94|6.84|
|  X10|2017.0|5.65|6.13|
|8Plus|2017.0|6.23|7.12|
+-----+------+----+----+



In [19]:
iphone_new = iphone_df.withColumn('price',F.when(F.col('Year') == 2017,50000).otherwise(80000))

In [20]:
iphone_new = iphone_df.withColumn("model_year",
                          F.expr("Model || '-' || Year "))

In [21]:
iphone_new.show()

+-----+----+------+-----+------+----------+
|Model|Year|Height|Width|Weight|model_year|
+-----+----+------+-----+------+----------+
|   XS|2018|  5.65| 2.79|  6.24|   XS-2018|
|   XR|2018|  5.94| 2.98|  6.84|   XR-2018|
|  X10|2017|  5.65| 2.79|  6.13|  X10-2017|
|8Plus|2017|  6.23| 3.07|  7.12|8Plus-2017|
+-----+----+------+-----+------+----------+



In [22]:
iphone_new.show()

+-----+----+------+-----+------+----------+
|Model|Year|Height|Width|Weight|model_year|
+-----+----+------+-----+------+----------+
|   XS|2018|  5.65| 2.79|  6.24|   XS-2018|
|   XR|2018|  5.94| 2.98|  6.84|   XR-2018|
|  X10|2017|  5.65| 2.79|  6.13|  X10-2017|
|8Plus|2017|  6.23| 3.07|  7.12|8Plus-2017|
+-----+----+------+-----+------+----------+



In [23]:
iphone_dropped = iphone_new.drop('Width','price')

In [24]:
iphone_dropped.show()

+-----+----+------+------+----------+
|Model|Year|Height|Weight|model_year|
+-----+----+------+------+----------+
|   XS|2018|  5.65|  6.24|   XS-2018|
|   XR|2018|  5.94|  6.84|   XR-2018|
|  X10|2017|  5.65|  6.13|  X10-2017|
|8Plus|2017|  6.23|  7.12|8Plus-2017|
+-----+----+------+------+----------+



Schema Writing as normal string

In [25]:
schema_str = 'Model string,year int'

from pyspark.sql.types import _parse_datatype_string
schema_spark = _parse_datatype_string(schema_str)
schema_spark

StructType([StructField('Model', StringType(), True), StructField('year', IntegerType(), True)])